In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    matthews_corrcoef,
    cohen_kappa_score,
    classification_report,
    confusion_matrix
)

In [ ]:
rq1_data = pd.read_csv("rq1_qualification_relevance_part1_dataset_v2.csv")
rq2_data = pd.read_csv("rq2_migrant_income_part1_dataset_v2.csv")

print(rq1_data.shape)
print(rq2_data.shape)

In [ ]:
categorical_filler = SimpleImputer(strategy="most_frequent")
numerical_filler = SimpleImputer(strategy="median")

In [ ]:
rq1_predictors = [
    "employment_status",
    "sex",
    "age_group",
    "age_midpoint",
    "number_of_non_school_qualifications",
    "qualification_recency",
    "current_job_skill_level",
    "country_of_birth_group",
    "citizenship_status",
    "occupation",
    "occupation_earners_2022_23",
    "occupation_median_age_2022_23",
    "occupation_sum_income_2022_23",
    "occupation_median_income_2022_23",
    "occupation_mean_income_2022_23",
    "occupation_income_level",
    "occupation_income_rank"
]

rq1_features = rq1_data[rq1_predictors].copy()
rq1_target = rq1_data["target"].copy()

rq1_cat_cols = rq1_features.select_dtypes(include="object").columns
rq1_num_cols = rq1_features.select_dtypes(exclude="object").columns

rq1_features.loc[:, rq1_cat_cols] = categorical_filler.fit_transform(rq1_features[rq1_cat_cols])
rq1_features.loc[:, rq1_num_cols] = numerical_filler.fit_transform(rq1_features[rq1_num_cols])

In [ ]:
rq1_encoders = {}

for column_name in rq1_cat_cols:
    encoder = LabelEncoder()
    rq1_features.loc[:, column_name] = encoder.fit_transform(rq1_features[column_name].astype(str))
    rq1_encoders[column_name] = encoder

rq1_target_encoder = LabelEncoder()
rq1_target_encoded = rq1_target_encoder.fit_transform(rq1_target)

In [ ]:
rq1_X_train, rq1_X_test, rq1_y_train, rq1_y_test = train_test_split(
    rq1_features,
    rq1_target_encoded,
    test_size=0.20,
    random_state=42,
    stratify=rq1_target_encoded
)

rq1_scaler = StandardScaler()
rq1_X_train_scaled = rq1_scaler.fit_transform(rq1_X_train)
rq1_X_test_scaled = rq1_scaler.transform(rq1_X_test)
rq1_features_scaled = rq1_scaler.fit_transform(rq1_features)

In [ ]:
random_forest_search_space = {
    "n_estimators": [100, 200],
    "max_depth": [5, 10, None]
}

random_forest_base = RandomForestClassifier(random_state=42)

random_forest_grid = GridSearchCV(
    estimator=random_forest_base,
    param_grid=random_forest_search_space,
    cv=5,
    scoring="f1_weighted"
)

random_forest_grid.fit(rq1_X_train, rq1_y_train)

random_forest_model = random_forest_grid.best_estimator_
random_forest_predictions = random_forest_model.predict(rq1_X_test)
random_forest_probabilities = random_forest_model.predict_proba(rq1_X_test)[:, 1]

print(random_forest_grid.best_params_)

In [ ]:
svm_search_space = {
    "C": [0.1, 1],
    "gamma": ["scale", 0.1]
}

svm_base = SVC(
    kernel="rbf",
    probability=True,
    random_state=42
)

svm_grid = GridSearchCV(
    estimator=svm_base,
    param_grid=svm_search_space,
    cv=5,
    scoring="f1_weighted"
)

svm_grid.fit(rq1_X_train_scaled, rq1_y_train)

svm_model = svm_grid.best_estimator_
svm_predictions = svm_model.predict(rq1_X_test_scaled)
svm_probabilities = svm_model.predict_proba(rq1_X_test_scaled)[:, 1]

print(svm_grid.best_params_)

In [ ]:
rq2_predictors = [
    "visa_group",
    "age_range",
    "applicant_status",
    "sex",
    "arrival_group",
    "age_midpoint"
]

rq2_features = rq2_data[rq2_predictors].copy()
rq2_target = rq2_data["income_target"].copy()

rq2_cat_cols = rq2_features.select_dtypes(include="object").columns
rq2_num_cols = rq2_features.select_dtypes(exclude="object").columns

rq2_features.loc[:, rq2_cat_cols] = categorical_filler.fit_transform(rq2_features[rq2_cat_cols])
rq2_features.loc[:, rq2_num_cols] = numerical_filler.fit_transform(rq2_features[rq2_num_cols])

In [ ]:
rq2_encoders = {}

for column_name in rq2_cat_cols:
    encoder = LabelEncoder()
    rq2_features.loc[:, column_name] = encoder.fit_transform(rq2_features[column_name].astype(str))
    rq2_encoders[column_name] = encoder

rq2_target_encoder = LabelEncoder()
rq2_target_encoded = rq2_target_encoder.fit_transform(rq2_target)

In [ ]:
rq2_X_train, rq2_X_test, rq2_y_train, rq2_y_test = train_test_split(
    rq2_features,
    rq2_target_encoded,
    test_size=0.20,
    random_state=42,
    stratify=rq2_target_encoded
)

rq2_scaler = StandardScaler()
rq2_X_train_scaled = rq2_scaler.fit_transform(rq2_X_train)
rq2_X_test_scaled = rq2_scaler.transform(rq2_X_test)
rq2_features_scaled = rq2_scaler.fit_transform(rq2_features)

In [ ]:
decision_tree_search_space = {
    "max_depth": [3, 5, 10],
    "min_samples_split": [2, 5]
}

decision_tree_base = DecisionTreeClassifier(random_state=42)

decision_tree_grid = GridSearchCV(
    estimator=decision_tree_base,
    param_grid=decision_tree_search_space,
    cv=5,
    scoring="f1"
)

decision_tree_grid.fit(rq2_X_train, rq2_y_train)

decision_tree_model = decision_tree_grid.best_estimator_
decision_tree_predictions = decision_tree_model.predict(rq2_X_test)
decision_tree_probabilities = decision_tree_model.predict_proba(rq2_X_test)[:, 1]

print(decision_tree_grid.best_params_)

In [ ]:
gaussian_nb_model = GaussianNB()

gaussian_nb_model.fit(rq2_X_train_scaled, rq2_y_train)

gaussian_nb_predictions = gaussian_nb_model.predict(rq2_X_test_scaled)
gaussian_nb_probabilities = gaussian_nb_model.predict_proba(rq2_X_test_scaled)[:, 1]

In [ ]:
def binary_model_summary(model_name, research_question, y_true, y_pred, y_score):
    return {
        "Research Question": research_question,
        "Model": model_name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Balanced Accuracy": balanced_accuracy_score(y_true, y_pred),
        "Precision Weighted": precision_score(y_true, y_pred, average="weighted", zero_division=0),
        "Recall Weighted": recall_score(y_true, y_pred, average="weighted", zero_division=0),
        "F1 Weighted": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "Precision Macro": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "Recall Macro": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "F1 Macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "ROC-AUC": roc_auc_score(y_true, y_score),
        "MCC": matthews_corrcoef(y_true, y_pred),
        "Cohen Kappa": cohen_kappa_score(y_true, y_pred)
    }

In [ ]:
model_performance = pd.DataFrame([
    binary_model_summary(
        "Random Forest",
        "RQ1",
        rq1_y_test,
        random_forest_predictions,
        random_forest_probabilities
    ),
    binary_model_summary(
        "Support Vector Machine",
        "RQ1",
        rq1_y_test,
        svm_predictions,
        svm_probabilities
    ),
    binary_model_summary(
        "Decision Tree",
        "RQ2",
        rq2_y_test,
        decision_tree_predictions,
        decision_tree_probabilities
    ),
    binary_model_summary(
        "Gaussian Naive Bayes",
        "RQ2",
        rq2_y_test,
        gaussian_nb_predictions,
        gaussian_nb_probabilities
    )
])

model_performance.round(4)

In [ ]:
random_forest_cv_accuracy = cross_val_score(
    random_forest_model,
    rq1_features,
    rq1_target_encoded,
    cv=5,
    scoring="accuracy"
)

svm_cv_accuracy = cross_val_score(
    svm_model,
    rq1_features_scaled,
    rq1_target_encoded,
    cv=5,
    scoring="accuracy"
)

decision_tree_cv_accuracy = cross_val_score(
    decision_tree_model,
    rq2_features,
    rq2_target_encoded,
    cv=5,
    scoring="accuracy"
)

gaussian_nb_cv_accuracy = cross_val_score(
    gaussian_nb_model,
    rq2_features_scaled,
    rq2_target_encoded,
    cv=5,
    scoring="accuracy"
)

In [ ]:
cross_validation_results = pd.DataFrame({
    "Research Question": ["RQ1", "RQ1", "RQ2", "RQ2"],
    "Model": [
        "Random Forest",
        "Support Vector Machine",
        "Decision Tree",
        "Gaussian Naive Bayes"
    ],
    "CV Accuracy Mean": [
        random_forest_cv_accuracy.mean(),
        svm_cv_accuracy.mean(),
        decision_tree_cv_accuracy.mean(),
        gaussian_nb_cv_accuracy.mean()
    ],
    "CV Accuracy Standard Deviation": [
        random_forest_cv_accuracy.std(),
        svm_cv_accuracy.std(),
        decision_tree_cv_accuracy.std(),
        gaussian_nb_cv_accuracy.std()
    ]
})

cross_validation_results.round(4)

In [ ]:
print("Random Forest Classification Report")
print(classification_report(rq1_y_test, random_forest_predictions, target_names=rq1_target_encoder.classes_))

print("Support Vector Machine Classification Report")
print(classification_report(rq1_y_test, svm_predictions, target_names=rq1_target_encoder.classes_))

print("Decision Tree Classification Report")
print(classification_report(rq2_y_test, decision_tree_predictions, target_names=rq2_target_encoder.classes_))

print("Gaussian Naive Bayes Classification Report")
print(classification_report(rq2_y_test, gaussian_nb_predictions, target_names=rq2_target_encoder.classes_))

In [ ]:
random_forest_matrix = confusion_matrix(rq1_y_test, random_forest_predictions)
svm_matrix = confusion_matrix(rq1_y_test, svm_predictions)
decision_tree_matrix = confusion_matrix(rq2_y_test, decision_tree_predictions)
gaussian_nb_matrix = confusion_matrix(rq2_y_test, gaussian_nb_predictions)

print("Random Forest Confusion Matrix")
print(random_forest_matrix)

print("Support Vector Machine Confusion Matrix")
print(svm_matrix)

print("Decision Tree Confusion Matrix")
print(decision_tree_matrix)

print("Gaussian Naive Bayes Confusion Matrix")
print(gaussian_nb_matrix)

In [ ]:
final_comparison = model_performance.merge(
    cross_validation_results,
    on=["Research Question", "Model"],
    how="left"
)

final_comparison = final_comparison[
    [
        "Research Question",
        "Model",
        "Accuracy",
        "Balanced Accuracy",
        "Precision Weighted",
        "Recall Weighted",
        "F1 Weighted",
        "F1 Macro",
        "ROC-AUC",
        "MCC",
        "Cohen Kappa",
        "CV Accuracy Mean",
        "CV Accuracy Standard Deviation"
    ]
]

final_comparison.round(4)

In [1]:
final_comparison.to_csv("section_6_model_evaluation_results.csv", index=False)

pd.DataFrame(random_forest_matrix).to_csv("rq1_random_forest_confusion_matrix.csv", index=False)
pd.DataFrame(svm_matrix).to_csv("rq1_svm_confusion_matrix.csv", index=False)
pd.DataFrame(decision_tree_matrix).to_csv("rq2_decision_tree_confusion_matrix.csv", index=False)
pd.DataFrame(gaussian_nb_matrix).to_csv("rq2_gaussian_nb_confusion_matrix.csv", index=False)

NameError: name 'final_comparison' is not defined